# 영화 추천 챗봇

사용자의 취향을 기억하는 영화 추천 챗봇 (OpenAI Agents SDK + Clean Architecture)

**기능:**
- 사용자가 좋아하는 장르를 기억
- 이미 시청한 영화를 기억
- 대화 기록 기반으로 개인화된 추천 제공

**구현 방식:**
- `messages` 리스트로 대화 히스토리 관리
- user/assistant 메시지를 모두 append
- 전체 대화 기록을 Agent에 전달하여 컨텍스트 유지

## 1. Domain Layer — 엔티티

In [1]:
from dataclasses import dataclass, field


@dataclass
class Message:
    """대화 메시지 엔티티."""
    role: str  # "user" | "assistant"
    content: str


@dataclass
class ConversationMemory:
    """대화 기록을 관리하는 엔티티.

    messages 리스트에 user/assistant 메시지를 축적하여
    에이전트가 이전 대화 컨텍스트를 활용할 수 있도록 한다.
    """
    messages: list[dict] = field(default_factory=list)

    def add_user_message(self, content: str) -> None:
        self.messages.append({"role": "user", "content": content})

    def add_assistant_message(self, content: str) -> None:
        self.messages.append({"role": "assistant", "content": content})

    def get_messages(self) -> list[dict]:
        return self.messages

## 2. Domain Layer — 포트 (인터페이스)

In [2]:
from abc import ABC, abstractmethod


class IChatAgent(ABC):
    """챗봇 에이전트 포트. Infrastructure 계층에서 구현한다."""

    @abstractmethod
    async def run(self, messages: list[dict]) -> str:
        """대화 기록을 받아 응답을 생성한다."""
        ...

## 3. Application Layer — Use Case

In [3]:
class ChatUseCase:
    """대화 유스케이스.

    ConversationMemory로 messages를 관리하고,
    IChatAgent 포트를 통해 에이전트 응답을 생성한다.
    """

    def __init__(self, agent: IChatAgent, memory: ConversationMemory):
        self._agent = agent
        self._memory = memory

    async def chat(self, user_input: str) -> str:
        # 1. 사용자 메시지 저장
        self._memory.add_user_message(user_input)

        # 2. 전체 대화 기록을 에이전트에 전달하여 응답 생성
        response = await self._agent.run(self._memory.get_messages())

        # 3. 에이전트 응답 저장
        self._memory.add_assistant_message(response)

        return response

## 4. Infrastructure Layer — OpenAI Agent 구현

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

from agents import Agent, Runner, RunConfig
from agents.models.openai_provider import OpenAIProvider

# 환경 설정
run_config = RunConfig(
    model_provider=OpenAIProvider(base_url=os.getenv("AI_BASE_URL")),
    tracing_disabled=True,
)

# 에이전트 정의
movie_chatbot_agent = Agent(
    name="Movie Recommendation Chatbot",
    model="gpt-4o-mini",
    instructions="""당신은 영화 추천 챗봇입니다.

대화를 통해 다음 정보를 기억하고 활용하세요:
1. 사용자가 좋아하는 장르
2. 사용자가 이미 본 영화

규칙:
- 이미 본 영화는 절대 추천하지 마세요.
- 추천 시 사용자의 취향(장르)을 반영하세요.
- 사용자가 이전 대화 내용을 물어보면 대화 기록을 기반으로 정확히 답변하세요.
- 항상 한국어로 친절하게 답변하세요.
- 답변은 간결하게 해주세요.""",
)


class OpenAIChatAgent(IChatAgent):
    """OpenAI Agents SDK 기반 IChatAgent 구현."""

    async def run(self, messages: list[dict]) -> str:
        result = await Runner.run(
            movie_chatbot_agent,
            messages,
            run_config=run_config,
        )
        return result.final_output

## 5. 조립 — 의존성 주입

In [5]:
# 의존성 조립
memory = ConversationMemory()
agent = OpenAIChatAgent()
chat_use_case = ChatUseCase(agent=agent, memory=memory)

## 6. 챗봇 실행

직접 대화를 입력하여 챗봇을 테스트합니다. 종료하려면 `q`를 입력하세요.

In [6]:
while True:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() == "q":
        print("종료합니다.")
        break
    response = await chat_use_case.chat(user_input)
    print(f"AI: {response}\n")

AI: SF 영화 좋아하시는군요! 이미 본 영화는 무엇인가요? 알려주시면 추천해드릴게요.

AI: 알겠습니다! 인셉션과 인터스텔라를 이미 보셨군요. 그럼 다음 영화로는 **"소스 코드"**나 **"제니시스"**를 추천합니다. 두 영화 모두 흥미로운 SF 요소가 가득해요!

AI: 물론이죠! 오늘 밤에는 **"클라우드 아틀라스"**를 추천합니다. 다양한 시간대의 이야기를 통해 깊이 있는 SF 요소를 느낄 수 있어요. 재미있게 보세요!

AI: 당신은 SF 영화를 좋아하시고, 이미 **인셉션**과 **인터스텔라**를 보셨다고 하셨어요. 맞나요?

종료합니다.


## 7. 대화 기록 확인

`messages` 리스트에 축적된 전체 대화 히스토리를 확인합니다.

In [7]:
for msg in memory.get_messages():
    role = "User" if msg["role"] == "user" else "AI"
    print(f"[{role}] {msg['content']}\n")

[User] 나는 SF 영화를 좋아해

[AI] SF 영화 좋아하시는군요! 이미 본 영화는 무엇인가요? 알려주시면 추천해드릴게요.

[User] 인셉션이랑 인터스텔라는 이미 봤어

[AI] 알겠습니다! 인셉션과 인터스텔라를 이미 보셨군요. 그럼 다음 영화로는 **"소스 코드"**나 **"제니시스"**를 추천합니다. 두 영화 모두 흥미로운 SF 요소가 가득해요!

[User] 오늘 밤에 뭐 볼지 추천해줄래?

[AI] 물론이죠! 오늘 밤에는 **"클라우드 아틀라스"**를 추천합니다. 다양한 시간대의 이야기를 통해 깊이 있는 SF 요소를 느낄 수 있어요. 재미있게 보세요!

[User] 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?

[AI] 당신은 SF 영화를 좋아하시고, 이미 **인셉션**과 **인터스텔라**를 보셨다고 하셨어요. 맞나요?

